# W02b.1 — LLM API Basics with LangChain

We'll start with the basics of LLM API calls with LangChain.

The rest of the notebook assumes you've installed the required packages and set up your .env file with the required keys as shown in class.

In [51]:
# import the API keys
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."
print("API key loaded")

API key loaded


## 1. Your first call

LangChain provides a small number of core features to get started with AI agents: `chat_models`, `messages`, `tools`, `agents`.

`init_chat_model` is the universal constructor for a chat model: give it a model name, get back a model with the same interface regardless of provider.

### Model Setup

Execute one of the following to define the model.

If using Ollama, you will need to start it first (simply open the chat UI and send a message):
- https://docs.langchain.com/oss/python/integrations/chat/ollama

Verify that Ollama is serving a model:
- http://localhost:11434/

In [52]:
# from langchain.chat_models import init_chat_model

# model = init_chat_model(model="gpt-4.1-mini")

In [53]:
# claude = init_chat_model(model="claude-sonnet-4-6")

In [54]:
# Google's own installation of the Gemini model, which is a bit different from the one in LangChain.  
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [55]:
# Ollama's own installation of the Qwen model, which is a bit different from the one in LangChain.
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=False)

### Sending an inference request

Use `invoke` to send a request to the model.

In [56]:
question = "In one sentence: What does temperature do in an API call?"

In [57]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv

load_dotenv() # loads Google API key from .env file

model=ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)
#TODO: send the request and store the response
#TODO: display the response
response = model.invoke(question)
print(response)

content=[{'type': 'text', 'text': "In an API call, temperature controls the level of randomness and creativity in the model's responses, where lower values yield more predictable and deterministic answers, and higher values produce more varied and creative outputs.", 'extras': {'signature': 'El4KXAFpFH0TWlrpojfjoYB7Ow5s3jnmpbmPruJcYQPpTWW7rr+42EFkHWs4svBTQlnTvfCLnFHsWYLrugYAt9ztzrIIadwQS7aa8QOtwHD35kl77ShQDyzU2Gi75Ne3'}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a0b646-613c-79a2-9533-4e9c3fa214b4-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 14, 'output_tokens': 40, 'total_tokens': 54, 'input_token_details': {'cache_read': 0}}


`invoke` returned an `AIMessage` object, not a string. It carries the text plus metadata about how it was produced. Look at what else is in there.

In [58]:
from pprint import pprint

print(type(response).__name__)
pprint(response.response_metadata)

AIMessage
{'finish_reason': 'STOP',
 'model_name': 'gemini-3.5-flash-lite',
 'model_provider': 'google_genai',
 'safety_ratings': []}


## 2. Messages and roles

APIs take a list of messages, each with a role:

| Role | Class | Who it is |
|---|---|---|
| system | `SystemMessage` | You, the developer: persona, rules |
| user | `HumanMessage` | The end user |
| assistant | `AIMessage` | The model's own past turns |

The system prompt is used to steer the assistant persona.

Example: `SystemMessage(content="your message")`

In [59]:
system_prompt = "You are a teaching assistant for an Agentic AI course. Answer in one sentence."
user_question = "What does temperature do in an API call?"

In [60]:
from langchain.messages import SystemMessage, HumanMessage

#TODO: define the list of messages containing system and user prompts.
def get_response(system_message, user_question):
    system_message = SystemMessage(content=system_message)
    human_message = HumanMessage(content=user_question)
    print(get_response.content)

    # messages = [system_message, human_message] # this is how context is added to a model
    # response = model.invoke(messages)
    # print(response.content)

## 3. The model is stateless

The model keeps no memory between calls. If you want a conversation, you must send the whole history every time. 

Let's see this in action. First turn:

In [61]:
from langchain.messages import AIMessage

conversation = [
    SystemMessage(content="You are a concise tutor."),
    HumanMessage(content="Hello from Boston. Give me a two-line analogy for what an embedding is."),
]

first_response = model.invoke(conversation)


In [62]:
type(first_response)

langchain_core.messages.ai.AIMessage

In [63]:
first_response

AIMessage(content=[{'type': 'text', 'text': 'Hello from Boston! \n\nAn embedding is like translating any word or object into coordinates on a giant map of meaning. \nJust as cities close on a map are near each other geographically, concepts with similar meanings end up close together in this numerical space.', 'extras': {'signature': 'El4KXAFpFH0TmVSSjdYjUPWiXbphLmlmfxdVd2PN5gOdDJSJyr11PoQ6FMxT2VTQusiqvEQc32Ga2hJ4aAvFI/ls2szXME8B+10Z2arvVALlQu2mfEsf4fa/6hAnqsDi'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0b646-6407-78b2-a711-8f2a2b05c61f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, 'output_tokens': 52, 'total_tokens': 76, 'input_token_details': {'cache_read': 0}})

In [64]:
next_message = HumanMessage(content="Where do I live?")
next_response = model.invoke([next_message])
print(next_response.content)

[{'type': 'text', 'text': "I don't know where you live. I don't have access to your personal information, location data, or IP address unless you explicitly share it with me.", 'extras': {'signature': 'El4KXAFpFH0TmGvHbAyHhmHR3x3HOlW8folH10y4VHyisChj2Ca5N+w+M5zE5+bC5mJkJnJnsH8zHinHJP5dHREUJOZjJYvtnEzeRk9Ax9vhooXGUqtr80PvAhyaGq2b'}}]


In [65]:
print(first_response.content)

[{'type': 'text', 'text': 'Hello from Boston! \n\nAn embedding is like translating any word or object into coordinates on a giant map of meaning. \nJust as cities close on a map are near each other geographically, concepts with similar meanings end up close together in this numerical space.', 'extras': {'signature': 'El4KXAFpFH0TmVSSjdYjUPWiXbphLmlmfxdVd2PN5gOdDJSJyr11PoQ6FMxT2VTQusiqvEQc32Ga2hJ4aAvFI/ls2szXME8B+10Z2arvVALlQu2mfEsf4fa/6hAnqsDi'}}]


To help the model remember previous conversations, we need to append the model's reply to the history:

In [66]:
pprint(conversation)

[SystemMessage(content='You are a concise tutor.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello from Boston. Give me a two-line analogy for what an embedding is.', additional_kwargs={}, response_metadata={})]


In [67]:
conversation.append(first_response)

In [68]:
pprint(conversation)

[SystemMessage(content='You are a concise tutor.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello from Boston. Give me a two-line analogy for what an embedding is.', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[{'type': 'text', 'text': 'Hello from Boston! \n\nAn embedding is like translating any word or object into coordinates on a giant map of meaning. \nJust as cities close on a map are near each other geographically, concepts with similar meanings end up close together in this numerical space.', 'extras': {'signature': 'El4KXAFpFH0TmVSSjdYjUPWiXbphLmlmfxdVd2PN5gOdDJSJyr11PoQ6FMxT2VTQusiqvEQc32Ga2hJ4aAvFI/ls2szXME8B+10Z2arvVALlQu2mfEsf4fa/6hAnqsDi'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0b646-6407-78b2-a711-8f2a2b05c61f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, '

In [69]:
model.invoke([next_message])

AIMessage(content=[{'type': 'text', 'text': "I don't know where you live. I don't have access to your personal information, location data, or IP address unless you explicitly share it with me!", 'extras': {'signature': 'El4KXAFpFH0TzcPlNzQ/ZGYaB8Pxby9ARjc7fNfEX5jq2S/eBk03oq1gHtVjtvubIz0BQx5zAW/oYzRLf4XtQ30FJ2DDGKJaKeE5dnUF0yIEZKA+bP+0vXr3lr5EusA+'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0b646-6948-70c1-b1a9-3fbec38e840a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 34, 'total_tokens': 40, 'input_token_details': {'cache_read': 0}})

We need to append the follow-up to the original list of messages and send everything again:

In [70]:
conversation

[SystemMessage(content='You are a concise tutor.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello from Boston. Give me a two-line analogy for what an embedding is.', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[{'type': 'text', 'text': 'Hello from Boston! \n\nAn embedding is like translating any word or object into coordinates on a giant map of meaning. \nJust as cities close on a map are near each other geographically, concepts with similar meanings end up close together in this numerical space.', 'extras': {'signature': 'El4KXAFpFH0TmVSSjdYjUPWiXbphLmlmfxdVd2PN5gOdDJSJyr11PoQ6FMxT2VTQusiqvEQc32Ga2hJ4aAvFI/ls2szXME8B+10Z2arvVALlQu2mfEsf4fa/6hAnqsDi'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0b646-6407-78b2-a711-8f2a2b05c61f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, '

In [71]:
conversation.append(next_message)

In [72]:
second_message = model.invoke(conversation)
print(second_message.content)

[{'type': 'text', 'text': 'Based on your previous message, you live in Boston.', 'extras': {'signature': 'El4KXAFpFH0TnXeERAlEW5RKMOCrSsW9FDmV6Cb8Zk/2NG7GM6169N7IeVd8EsAr3KAd5PbyvhA1SUQwwzG6Xk7DRW2kJtJGozE6JfRNPtbvY5hL8ISpFgIwGYtjJ4S+'}}]


In [73]:
new_message = HumanMessage("Give me another analogy that's more relatable to someone who loves cooking.")
conversation.append(new_message)

In [74]:
last_response = model.invoke(conversation)

print(last_response.content)

[{'type': 'text', 'text': "Based on your greeting, you live in Boston!\n\nAn embedding is like a chef's flavor profile, breaking down any ingredient into scores for sweetness, umami, spice, and earthiness. \nJust as cinnamon and nutmeg end up clustered near each other because they taste similar, similar concepts end up with close coordinate values.", 'extras': {'signature': 'El4KXAFpFH0TXNJ4FSP8lHY0924bX/7qvLimiGfb2REkLzilVnRnRkoKMiEudqPLi0RmKOUnOOQQx4eVcINAZVA/UGMyfWDZKNJzZ5GUpI97OUdDWdHJcM/EHdHAzfp9'}}]


In [75]:
pprint(conversation)

[SystemMessage(content='You are a concise tutor.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello from Boston. Give me a two-line analogy for what an embedding is.', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[{'type': 'text', 'text': 'Hello from Boston! \n\nAn embedding is like translating any word or object into coordinates on a giant map of meaning. \nJust as cities close on a map are near each other geographically, concepts with similar meanings end up close together in this numerical space.', 'extras': {'signature': 'El4KXAFpFH0TmVSSjdYjUPWiXbphLmlmfxdVd2PN5gOdDJSJyr11PoQ6FMxT2VTQusiqvEQc32Ga2hJ4aAvFI/ls2szXME8B+10Z2arvVALlQu2mfEsf4fa/6hAnqsDi'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0b646-6407-78b2-a711-8f2a2b05c61f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, '

Every extra turn grows the context the model re-reads. This is why long agent sessions get expensive.

## 4. The sampling knobs

As we discussed, the model outputs a probability distribution over the next token. The API knobs decide how we draw from it.

- `temperature` divides the logits before softmax. Low sharpens (more deterministic), high flattens.
- `top_p` keeps only the smallest set of tokens covering probability mass p.
- `max_tokens` is a hard cap on output length.

BUT: reasoning-first models (the `gpt-5` family) fix their sampling internally. For these experiments we'll use a standard chat model without reasoning.

In [76]:
#sampler = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")
sampler = ChatOllama(model="qwen3.5:4b", reasoning=False)
#sampler = init_chat_model(model="gpt-4.1-mini")


Sampling is random, so a single call proves nothing. A small helper to run the same prompt several times:

In [77]:
def sample_n(prompt, n=4, **model_kwargs):
    #m = init_chat_model(model="gpt-4.1-mini", **model_kwargs)
    m = ChatOllama(model="qwen3.5:4b", reasoning=False, **model_kwargs)
    return [m.invoke(prompt).content.strip() for _ in range(n)]

prompt = "Invent a name for Northeastern University's mascot, husky. Reply with the name only."


In [78]:
print("temperature = 0.0  (near-greedy)")

for out in sample_n(prompt, temperature=0.0):
    print(" ", out)

temperature = 0.0  (near-greedy)
  The Husk
  The Husk
  The Husk
  The Husk


In [79]:
print("temperature = 1.4  (flattened distribution)")
for out in sample_n(prompt, temperature=1.4):
    print(" ", out)

temperature = 1.4  (flattened distribution)
  The Northeast Husk
  Husky Bear
  Puckers
  Nor'Easter


Near-identical names at 0.0, real variety at 1.4. Two caveats: even at temperature 0 the answers can occasionally differ, and high temperature is not more intelligence, only a flatter distribution.

In [80]:
print("top_p = 0.1  (only the most probable tokens survive)")
for out in sample_n(prompt, top_p=0.1):
    print(" ", out)

top_p = 0.1  (only the most probable tokens survive)
  The Husk
  The Husk
  The Husk
  The Husk


### Choosing settings:

| Task | Setting |
|---|---|
| Tool calls, structured output | temperature 0 to 0.3 |
| Factual Q&A, extraction | low 0.3 to 0.5 |
| Brainstorming, drafts | 0.8 to 1.2, several samples |
| General chat | defaults; tune one knob, not both |

**Rule of thumb for agents: agents loop, and a small weirdness rate per step compounds across twenty steps. Keep agent temperature low.**

### max_tokens
It's also possible to limit the number of tokens to be generated. Use `max_tokens` as a cost ceiling. 

In [81]:
#capped = ChatOllama(model="qwen3.5:4b", reasoning=False, max_tokens = 50)
capped = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", max_tokens=50)
r = capped.invoke("Explain how backpropagation works.")


In [82]:
pprint(r.content)
pprint("\nstop reason:", r.response_metadata.get("finish_reason") or r.response_metadata.get("stop_reason"))

[{'extras': {'signature': 'El4KXAFpFH0T0zifN5adzvgxqTV4jN2n3RSroi/KCF6owcgzPxca8FkVra/NSzxXTmuivHO3rFNIARXYEBPtj6ceY3B7jVCkQGTJIziesmteTmB4RxOl4HVP7ZEzo9yy'},
  'text': 'At its core, **backpropagation** (short for "backward propagation '
          'of errors") is the central mathematical mechanism that allows '
          'artificial neural networks to learn. \n'
          '\n'
          'If a neural network is like a student taking a test, **forward',
  'type': 'text'}]


AttributeError: 'str' object has no attribute 'write'

The text stops mid-sentence and the stop reason says `length`, not `stop`. 

## 5. Streaming

Generation is one token per forward pass. Streaming exposes that rhythm instead of making you wait for the full reply.

Use `model.stream()` to stream each chunk as you receive them. Will need to iterate through the 
- model.stream("your query") will return the response as a sequence

In [ ]:
#TODO: iterate through the stream
#TODO: display the chunks

model.stream("Write me a song for Northeastern University about the husky mascot.")
for chunk in model.stream("Write a poem about the husky mascot.", stream=True):
    print(chunk.content, end="")

NameError: name 'model' is not defined